# Protocol 1 — EEG Interoceptive Precision Gating

Synthetic walkthrough of Protocol 1 (*EEG Study of Interoceptive Precision Gating*), updated to
**protocol specification v1.1.0**. A within-participant attention × cardiac-phase detection design tests how
interoceptive precision $\Pi^i_{\text{eff}}$ gates ignition of near-threshold visual stimuli.

Prediction cluster **Pred 1.A–1.F** (confirmatory α = 0.0125 unless stated):

* **Pred 1.A** — Ordered P3b (250–500 ms, Pz/P3/P4): interoceptive > exteroceptive > dual-task;
  ηₚ² ≥ 0.06 and |interoceptive − exteroceptive| ≥ 0.3 µV. *Falsified*: ηₚ² < 0.02, or |Δ| < 0.3 µV with BF₀₁ ≥ 6.
* **Pred 1.B** — Diastole vs. systole hit-rate advantage ≥ 5 percentage points (CardiacPhase × Condition interaction); d′ higher at diastole.
* **Pred 1.C** — Continuous interoceptive accuracy moderates the condition effect on P3b (IA_z × condition, β_std ≥ 0.30).
  *Falsified*: β_std < 0.10 with BF₀₁ ≥ 6. Tertile ANOVA is descriptive only.
* **Pred 1.D** — HEP predicts P3b beyond arousal (partial r > 0.40 after pupil/HRV control). *Falsified*: partial r < 0.20.
* **Pred 1.E** *(estimation, no α)* — Weibull shape k ≡ α_psy on linear contrast: 95 % interval lower bound ≥ 10 **and**
  graded model loses by ΔAIC > 10. *Against*: upper bound < 5 or graded model wins by ΔAIC > 10.
* **Pred 1.F** *(secondary, α = 0.05)* — Trait IA moderates interoceptive-condition d′ (IA_z × Condition β_std ≥ 0.36).

> **Note:** Synthetic data mirror the v1.1 design. Πⁱ phase factors, attention gains and the contrast-to-error
> mapping are illustrative generative choices, not registered values. Replace the simulator with your loader
> once real data exist.
>
> **EP-1 | Paper 1 | Empirical | Depends on: EP-0 | Spec v1.1.0, pre-collection | N enrolled = 62, analysable target = 59, conditional floor = 48**

### What changed from v1.0

| Item | v1.0 notebook | v1.1 specification |
|---|---|---|
| N | 40 target / 32 minimum | 62 enrolled, 59 analysable (80 % power), 48 only under BF₁₀ ≥ 100 efficacy stop |
| Design | 480 trials, two phases | 3 attention sessions × 480 main trials (360 signal at six c50 multiples + 120 catch) |
| Cardiac windows | Fixed systole/diastole | Personalized: systole R+250 ms → T-end; diastole T-end+100 ms → next R−100 ms, prospectively balanced |
| α | 0.05 / 0.017 | 0.0125 for Pred 1.A–1.D |
| IA moderation | Tertile split (1.C) | Continuous IA_z × condition; tertiles descriptive |
| New | — | Pred 1.E Weibull steepness; Pred 1.F trait effect (was 1.A-trait) |
| κ, α, β | 100, 0.3, 0.7 | κ = `null` (uncalibrated); no legacy α/β defaults |

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import integrate
from scipy.optimize import minimize
from scipy.stats import norm, pearsonr, ttest_1samp, ttest_rel

from apgi.clinical import EnhancedClinicalInterpreter
from apgi.core import BETA_SM_DEFAULT, GAMMA_SIG_DEFAULT, compute_pi_i_eff, ignition_probability

In [ ]:
# Load the v1.1 protocol specification so every threshold below is read from
# the registered JSON rather than retyped by hand.
def load_protocol(filename):
    for root in (Path.cwd(), *Path.cwd().parents):
        path = root / "protocols" / filename
        if path.exists():
            return json.loads(path.read_text(encoding="utf-8"))
    raise FileNotFoundError(f"protocols/{filename} not found above {Path.cwd()}")


PROTO = load_protocol("protocol_1_cardiac_eeg.json")
assert PROTO["version"] == "1.1.0", PROTO["version"]
PRED = {p["id"]: p for p in PROTO["sub_predictions"]}
print(f"{PROTO['protocol_id']}  spec v{PROTO['version']}  status={PROTO['study_status']}")
for pid, p in PRED.items():
    print(f"  {pid:<9} [{p['status']}]  {p['claim'][:78]}")


def verdict(confirmed, falsified=False):
    """v1.1 three-way decision: failing to confirm is not falsification."""
    if confirmed:
        return "CONFIRMED (synthetic)"
    if falsified:
        return "FALSIFIED (synthetic)"
    return "INDETERMINATE (not confirmed; falsification rule not met)"

In [ ]:
def jzs_bf10(t, n, r=0.707):
    """JZS Bayes factor for a one-sample / paired t-test (Rouder et al., 2009).

    Cauchy prior scale r = 0.707, as registered for the v1.1 Bayesian tests.
    The likelihood ratio is formed inside the integral to avoid underflow.
    """
    nu = n - 1

    def integrand(g):
        ratio = ((1 + t**2 / ((1 + n * g * r**2) * nu)) / (1 + t**2 / nu)) ** (-(nu + 1) / 2)
        prior = (2 * np.pi) ** -0.5 * g**-1.5 * np.exp(-1 / (2 * g))
        return (1 + n * g * r**2) ** -0.5 * ratio * prior

    return float(integrate.quad(integrand, 0, np.inf, limit=200)[0])

## 1 — Protocol 1 design (v1.1)

Each participant completes one session per attention condition. A session holds 480 main trials:
360 signal trials (60 at each of six fixed c50 multiples) and 120 catch trials, with cardiac phase
prospectively balanced 50/50. The separate 80-trial mini-block and the unresolved QUEST top-up (30 or 50 trials)
are not simulated.

In [ ]:
BLOCKS = PROTO["blocks"]
TRIALS = PROTO["trial_structure"]
N_ENROLLED = PROTO["participants"]["n_target"]            # 62
N_ANALYSABLE = PROTO["participants"]["n_analyzable_target"]  # 59
N_FLOOR = PROTO["participants"]["n_minimum"]               # 48

CONDITIONS = TRIALS["conditions"]                    # interoceptive_focus, exteroceptive_focus, dual_task
LEVELS = np.array(BLOCKS["contrast_multipliers_c50"])   # 0.5 … 1.5 × c50
NEAR = np.array(TRIALS["near_threshold_analysis_levels_c50"])
N_SIGNAL = BLOCKS["main_signal_trials_per_session"]     # 360
N_CATCH = BLOCKS["main_catch_trials_per_session"]       # 120
N_PER_LEVEL = N_SIGNAL // len(LEVELS)                   # 60 ≥ usable minimum of 40
assert N_SIGNAL + N_CATCH == BLOCKS["trials_per_block"]
assert N_PER_LEVEL >= BLOCKS["usable_signal_minimum_per_level"]

APGI = PROTO["apgi_parameters"]
KAPPA = APGI["kappa"]["value"]   # None — uncalibrated ATP/bit bridge
BETA_SM = BETA_SM_DEFAULT        # 0.60 ∈ typical range [0.3, 0.7]
GAMMA_SIG = GAMMA_SIG_DEFAULT    # 5.0 ∈ registered range [2, 7.5]; distinct from α_psy
assert APGI["gamma_sig"]["range"][0] <= GAMMA_SIG <= APGI["gamma_sig"]["range"][1]

# Illustrative generative values (synthetic; not registered)
PHASE_FACTOR = {"systole": 0.6, "diastole": 1.2}   # baroreceptor gating of Πⁱ
ATTENTION_GAIN = {                                  # (exteroceptive gain, interoceptive gain)
    "interoceptive_focus": (1.00, 1.80),
    "exteroceptive_focus": (1.00, 0.60),
    "dual_task": (0.92, 0.85),
}
Z_CONTRAST = 3.0   # exteroceptive error per c50 unit
W_INTERO = 0.25    # relative interoceptive gain (calibration-fixed in v1.1)
IA_GAIN = 0.35     # IA_z moderation of interoceptive attention gain

print(f"Sessions: {len(CONDITIONS)} × {BLOCKS['trials_per_block']} main trials "
      f"({N_PER_LEVEL} per contrast level, {N_CATCH} catch)")
print("Systole :", TRIALS["cardiac_phase"]["systole"])
print("Diastole:", TRIALS["cardiac_phase"]["diastole"])
print(f"κ = {KAPPA}, β_SM = {BETA_SM}, γ_sig = {GAMMA_SIG}")

## 2 — Simulate the cohort

In [ ]:
def simulate_participant(ia_z, rng):
    """Return trial arrays for the three attention sessions of one participant."""
    pi_i_base = float(np.exp(0.15 * ia_z))       # trait interoceptive precision
    pi_e = float(rng.lognormal(0.0, 0.1))
    # QUEST anchors c50 at 50 % detection in the exteroceptive, phase-averaged state
    g_i_ext = ATTENTION_GAIN["exteroceptive_focus"][1]
    theta = pi_e * Z_CONTRAST + W_INTERO * pi_i_base * g_i_ext * 0.9 * np.sqrt(2 / np.pi)
    out = {}
    for cond in CONDITIONS:
        g_e, g_i = ATTENTION_GAIN[cond]
        if cond == "interoceptive_focus":
            g_i *= max(0.1, 1 + IA_GAIN * ia_z)
        contrast = np.concatenate([np.repeat(LEVELS, N_PER_LEVEL), np.zeros(N_CATCH)])
        rng.shuffle(contrast)
        phase = np.tile(["systole", "diastole"], len(contrast) // 2)  # prospective 50/50 balance
        rng.shuffle(phase)
        M_hat = rng.uniform(0.0, 0.5, len(contrast))
        pi_i_eff = np.array([compute_pi_i_eff(pi_i_base * PHASE_FACTOR[ph] * g_i, BETA_SM, float(m))
                             for ph, m in zip(phase, M_hat)])
        z_e = Z_CONTRAST * contrast * np.abs(1 + 0.1 * rng.normal(size=len(contrast)))
        z_i = np.abs(rng.normal(size=len(contrast)))
        S = pi_e * g_e * z_e + W_INTERO * pi_i_eff * z_i
        p_ign = np.array([ignition_probability(float(s), theta, GAMMA_SIG) for s in S])
        detected = rng.random(len(S)) < p_ign
        offset = rng.normal(0, 0.8)  # participant × session P3b offset (µV)
        p3b = 1.0 + offset + detected * (4.0 + 2.0 * (S - theta)) + rng.normal(0, 2.0, len(S))  # µV
        out[cond] = dict(contrast=contrast, phase=phase, detected=detected, S=S, theta=theta,
                         p3b=p3b, pi_i_eff=pi_i_eff)
    return out, pi_i_base


rng = np.random.default_rng(2025)
IA_z = rng.normal(0, 1, N_ENROLLED)
cohort, pi_i_trait = [], []
for ia in IA_z:
    sessions, pib = simulate_participant(float(ia), rng)
    cohort.append(sessions)
    pi_i_trait.append(pib)
pi_i_trait = np.array(pi_i_trait)

hit_by_level = {c: np.array([[s[c]["detected"][s[c]["contrast"] == lv].mean() for lv in LEVELS]
                             for s in cohort]) for c in CONDITIONS}
for c in CONDITIONS:
    print(f"{c:<20} hit rate by c50 multiple:", np.round(hit_by_level[c].mean(0), 2))

## 3 — Prediction 1.B: Cardiac-phase detection advantage

In [ ]:
def near_mask(s):
    return np.isin(s["contrast"], NEAR)


def phase_rates(s, phase):
    m = s["phase"] == phase
    hit = s["detected"][near_mask(s) & m].mean()
    fa = s["detected"][(s["contrast"] == 0) & m].mean()
    n_sig, n_fa = (near_mask(s) & m).sum(), ((s["contrast"] == 0) & m).sum()
    d = norm.ppf((hit * n_sig + 0.5) / (n_sig + 1)) - norm.ppf((fa * n_fa + 0.5) / (n_fa + 1))
    return hit, d


THR_B = PRED["Pred 1.B"]["confirming_thresholds"]
adv = {c: np.array([phase_rates(s[c], "diastole")[0] - phase_rates(s[c], "systole")[0] for s in cohort])
       for c in CONDITIONS}
dprime_adv = np.array([phase_rates(s["interoceptive_focus"], "diastole")[1]
                       - phase_rates(s["interoceptive_focus"], "systole")[1] for s in cohort])

# CardiacPhase × Condition: phase advantage in interoceptive vs exteroceptive focus
t_int, p_int = ttest_rel(adv["interoceptive_focus"], adv["exteroceptive_focus"])
adv_pp = adv["interoceptive_focus"].mean() * 100
confirmed_1B = (adv_pp >= THR_B["hit_rate_advantage_percentage_points_minimum"]
                and p_int < THR_B["alpha"] and dprime_adv.mean() > 0)

for c in CONDITIONS:
    print(f"{c:<20} diastole − systole hit rate = {adv[c].mean() * 100:+.1f} pp")
print(f"Interaction (intero vs extero phase advantage): t({N_ENROLLED - 1}) = {t_int:.2f}, p = {p_int:.2e}")
print(f"Δd′ (diastole − systole, interoceptive focus) = {dprime_adv.mean():+.2f}")
print(f"Pred 1.B → {verdict(confirmed_1B)}")

### Sequential monitoring (v1.1)

Interim looks at N = 30 (monitoring only), 50 and 60; efficacy stop requires BF₁₀ ≥ 100 for **every**
Pred 1.A–1.D at an eligible look. The cell below tracks the Pred 1.B interaction contrast only.

In [ ]:
SEQ = PROTO["sequential_design"]
contrast_1B = adv["interoceptive_focus"] - adv["exteroceptive_focus"]
for n in SEQ["interim_n"] + [N_ENROLLED]:
    t_n = ttest_1samp(contrast_1B[:n], 0.0).statistic
    role = "monitoring only" if n == 30 else ("final" if n == N_ENROLLED else "efficacy look")
    print(f"N = {n:>2} ({role:<15}) Pred 1.B BF₁₀ = {jzs_bf10(t_n, n):.3g}")
print("Futility rule:", SEQ["futility"])

## 4 — Prediction 1.A: Ordered P3b by attention condition

In [ ]:
THR_A = PRED["Pred 1.A"]["confirming_thresholds"]
p3b = np.array([[s[c]["p3b"][near_mask(s[c])].mean() for c in CONDITIONS] for s in cohort])  # (N, 3)

# One-way repeated-measures ANOVA, partial η²
grand = p3b.mean()
ss_cond = N_ENROLLED * ((p3b.mean(0) - grand) ** 2).sum()
ss_subj = len(CONDITIONS) * ((p3b.mean(1) - grand) ** 2).sum()
ss_err = ((p3b - grand) ** 2).sum() - ss_cond - ss_subj
eta_p2 = ss_cond / (ss_cond + ss_err)

_, p_ie = ttest_rel(p3b[:, 0], p3b[:, 1], alternative="greater")
_, p_ed = ttest_rel(p3b[:, 1], p3b[:, 2], alternative="greater")
delta_ie = (p3b[:, 0] - p3b[:, 1]).mean()
confirmed_1A = (eta_p2 >= THR_A["eta_p_squared_minimum"]
                and abs(delta_ie) >= THR_A["absolute_intero_minus_extero_uv_minimum"]
                and max(p_ie, p_ed) < THR_A["alpha"])
falsified_1A = eta_p2 < 0.02

print("Mean P3b (µV):", {c: round(float(v), 2) for c, v in zip(CONDITIONS, p3b.mean(0))})
print(f"ηₚ² = {eta_p2:.3f}; intero − extero = {delta_ie:.2f} µV (p = {p_ie:.2e}); "
      f"extero > dual p = {p_ed:.2e}")
print(f"Pred 1.A → {verdict(confirmed_1A, falsified_1A)}")

## 5 — Pred 1.C and Pred 1.F: Continuous interoceptive-accuracy moderation

In [ ]:
def beta_std(x, y):
    """Standardized slope of y on x (equals Pearson r for a single predictor)."""
    return float(np.corrcoef(x, y)[0, 1])


p3b_effect = p3b[:, 0] - p3b[:, 1]  # interoceptive − exteroceptive P3b
b_1C = beta_std(IA_z, p3b_effect)
p_1C = pearsonr(IA_z, p3b_effect)[1]
THR_C = PRED["Pred 1.C"]["confirming_thresholds"]
confirmed_1C = b_1C >= THR_C["continuous_ia_interaction_beta_std_minimum"] and p_1C < THR_C["alpha"]

def session_dprime(s):
    hit = s["detected"][near_mask(s)].mean()
    fa = s["detected"][s["contrast"] == 0].mean()
    n_sig, n_fa = near_mask(s).sum(), (s["contrast"] == 0).sum()
    return norm.ppf((hit * n_sig + 0.5) / (n_sig + 1)) - norm.ppf((fa * n_fa + 0.5) / (n_fa + 1))

dprime = np.array([[session_dprime(s[c]) for c in CONDITIONS] for s in cohort])
dprime_effect = dprime[:, 0] - dprime[:, 1]
b_1F = beta_std(IA_z, dprime_effect)
p_1F = pearsonr(IA_z, dprime_effect)[1]
THR_F = PRED["Pred 1.F"]["confirming_thresholds"]
confirmed_1F = b_1F >= THR_F["continuous_ia_interaction_beta_std_minimum"] and p_1F < THR_F["alpha"]

print(f"Pred 1.C  IA_z × condition on P3b: β_std = {b_1C:.2f}, p = {p_1C:.2e} → {verdict(confirmed_1C)}")
print(f"Pred 1.F  IA_z × condition on d′:  β_std = {b_1F:.2f}, p = {p_1F:.2e} → {verdict(confirmed_1F)}")

# Tertiles are descriptive only in v1.1
tert = np.digitize(IA_z, np.quantile(IA_z, [1 / 3, 2 / 3]))
for k, name in enumerate(["low", "mid", "high"]):
    print(f"  IA tertile {name:<4} (descriptive): P3b effect = {p3b_effect[tert == k].mean():+.2f} µV")

## 6 — Prediction 1.D: HEP predicts P3b beyond arousal

Participant-level HEP (interoceptive session) and P3b share an arousal component indexed by pupil
diameter and HRV. The registered statistic is the partial correlation after removing both covariates.

In [ ]:
arousal = rng.normal(0, 1, N_ENROLLED)
pupil = arousal + rng.normal(0, 0.5, N_ENROLLED)
hrv = -arousal + rng.normal(0, 0.5, N_ENROLLED)
pi_eff_intero = np.array([s["interoceptive_focus"]["pi_i_eff"].mean() for s in cohort])
hep = 0.10 + 0.80 * pi_eff_intero + 0.15 * arousal + rng.normal(0, 0.30, N_ENROLLED)
p3b_intero = p3b[:, 0] + 0.4 * arousal


def partial_r(x, y, covariates):
    Z = np.column_stack([np.ones(len(x))] + covariates)
    rx = x - Z @ np.linalg.lstsq(Z, x, rcond=None)[0]
    ry = y - Z @ np.linalg.lstsq(Z, y, rcond=None)[0]
    return pearsonr(rx, ry)


r_raw = pearsonr(hep, p3b_intero)[0]
r_part, p_part = partial_r(hep, p3b_intero, [pupil, hrv])
THR_D = PRED["Pred 1.D"]["confirming_thresholds"]
confirmed_1D = r_part > THR_D["partial_r_strict_minimum"] and p_part < THR_D["alpha"]
falsified_1D = r_part < 0.20
print(f"r(HEP, P3b) = {r_raw:.2f}; partial r | pupil, HRV = {r_part:.2f} (p = {p_part:.2e})")
print(f"Pred 1.D → {verdict(confirmed_1D, falsified_1D)}")

## 7 — Prediction 1.E: Weibull psychometric steepness (estimation)

$\psi(c) = g + (1-g-\lambda)\,\bigl(1 - e^{-(c/c_0)^k}\bigr)$ on linear contrast, with $g$ the empirical
catch false-alarm rate and lapse $\lambda \in [0, 0.05]$. Here $k \equiv \alpha_{psy}$, distinct from $\gamma_{sig}$.

> The registered analysis is a hierarchical beta-binomial model whose priors and graded comparator are still
> awaiting lock. This cell substitutes per-participant maximum-likelihood fits, a bootstrap interval on the
> group median, and a placeholder linear-ramp comparator.
>
> In this reduced single-window model the fitted k scales roughly with γ_sig × the exteroceptive error per
> c50 unit (here 5 × 3). The generator does not by itself guarantee the k ≥ 10 discrete-transition
> signature, so an INDETERMINATE result is an honest outcome of the illustrative scaling, not a bug.

In [ ]:
THR_E = PRED["Pred 1.E"]["confirming_thresholds"]
LAPSE_MAX = PRED["Pred 1.E"]["psychometric_model"]["lapse_range"][1]


def weibull(c, c0, k, lam, g):
    return g + (1 - g - lam) * (1 - np.exp(-(c / c0) ** k))


def ramp(c, c_sat, lam, g):
    return g + (1 - g - lam) * np.clip(c / c_sat, 0, 1)


def nll(pred, hits, n):
    pred = np.clip(pred, 1e-6, 1 - 1e-6)
    return -np.sum(hits * np.log(pred) + (n - hits) * np.log(1 - pred))


def fit_session(s):
    g = s["detected"][s["contrast"] == 0].mean()
    hits = np.array([s["detected"][s["contrast"] == lv].sum() for lv in LEVELS])
    n = np.full(len(LEVELS), N_PER_LEVEL)
    w = minimize(lambda x: nll(weibull(LEVELS, x[0], x[1], x[2], g), hits, n), x0=[1.0, 4.0, 0.01],
                 bounds=[(0.2, 3.0), (0.5, 60.0), (0.0, LAPSE_MAX)], method="L-BFGS-B")
    r = minimize(lambda x: nll(ramp(LEVELS, x[0], x[1], g), hits, n), x0=[1.5, 0.01],
                 bounds=[(0.5, 5.0), (0.0, LAPSE_MAX)], method="L-BFGS-B")
    return w.x, 2 * 3 + 2 * w.fun, 2 * 2 + 2 * r.fun


fits = [fit_session(s["exteroceptive_focus"]) for s in cohort]
k_hat = np.array([f[0][1] for f in fits])
delta_aic = sum(f[2] - f[1] for f in fits)  # AIC_graded − AIC_Weibull, summed over participants

boot_rng = np.random.default_rng(3)
boot_med = np.array([np.median(boot_rng.choice(k_hat, len(k_hat))) for _ in range(5000)])
k_lo, k_hi = np.percentile(boot_med, [2.5, 97.5])

confirmed_1E = k_lo >= THR_E["population_shape_hdi_lower_minimum"] and delta_aic > THR_E["delta_aic_strict_minimum"]
against_1E = (k_hi < 5) or (delta_aic < -10)
print(f"Group median k = {np.median(k_hat):.1f} (95% bootstrap interval {k_lo:.1f}–{k_hi:.1f})")
print(f"ΔAIC (graded − Weibull) = {delta_aic:.1f}")
print(f"Pred 1.E (estimation) → {verdict(confirmed_1E, against_1E)}")

## 8 — Visualize Pred 1.A, 1.B and 1.E

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
colors = {"interoceptive_focus": "#2166ac", "exteroceptive_focus": "#9966FF", "dual_task": "#AAAAAA"}
labels = {"interoceptive_focus": "Interoceptive", "exteroceptive_focus": "Exteroceptive", "dual_task": "Dual-task"}

ax = axes[0]
for c in CONDITIONS:
    ax.plot(LEVELS, hit_by_level[c].mean(0), "o-", color=colors[c], label=labels[c])
cc = np.linspace(0.4, 1.6, 200)
med_fit = np.median([f[0] for f in fits], axis=0)
g_ext = np.mean([s["exteroceptive_focus"]["detected"][s["exteroceptive_focus"]["contrast"] == 0].mean()
                 for s in cohort])
ax.plot(cc, weibull(cc, med_fit[0], med_fit[1], med_fit[2], g_ext), "k--", lw=1,
        label=f"Weibull fit (k = {med_fit[1]:.1f})")
ax.set_xlabel("Contrast (× c50)")
ax.set_ylabel("Hit rate")
ax.set_title("Psychometric functions (Pred 1.E)")
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[1]
x = np.arange(len(CONDITIONS))
means = [adv[c].mean() * 100 for c in CONDITIONS]
sems = [adv[c].std() / np.sqrt(N_ENROLLED) * 100 for c in CONDITIONS]
ax.bar(x, means, yerr=sems, color=[colors[c] for c in CONDITIONS], alpha=0.85, edgecolor="white",
       capsize=4, width=0.55)
ax.axhline(THR_B["hit_rate_advantage_percentage_points_minimum"], ls="--", lw=1, color="k",
           label="5 pp criterion")
ax.set_xticks(x, [labels[c] for c in CONDITIONS])
ax.set_ylabel("Diastole − systole hit rate (pp)")
ax.set_title(f"Pred 1.B — cardiac-phase advantage\ninteraction p = {p_int:.1e}")
ax.legend(fontsize=8)
ax.spines[["top", "right"]].set_visible(False)

ax = axes[2]
ax.bar(x, p3b.mean(0), yerr=p3b.std(0) / np.sqrt(N_ENROLLED), color=[colors[c] for c in CONDITIONS],
       alpha=0.85, edgecolor="white", capsize=4, width=0.55)
ax.set_xticks(x, [labels[c] for c in CONDITIONS])
ax.set_ylabel("Near-threshold P3b (µV)")
ax.set_title(f"Pred 1.A — ordered P3b\nηₚ² = {eta_p2:.2f}, Δ(int − ext) = {delta_ie:.2f} µV")
ax.spines[["top", "right"]].set_visible(False)

fig.suptitle(f"Protocol 1 — Interoceptive precision gating · spec v1.1.0 · synthetic N = {N_ENROLLED}", fontsize=12)
plt.tight_layout()
plt.show()

## 9 — Clinical interpretation with `EnhancedClinicalInterpreter`

Slide a 20-trial window over one participant's interoceptive-focus session and label each window.

In [ ]:
session = cohort[0]["interoceptive_focus"]
theta_series = np.full(len(session["S"]), session["theta"])
C_metabolic = rng.uniform(0.5, 2.0, len(session["S"]))  # illustrative metabolic-cost trace

interp = EnhancedClinicalInterpreter(window_size=20)
reports = interp.interpret_session(session["S"], theta_series, C_metabolic)
levels, counts = np.unique([r.level.value for r in reports], return_counts=True)
for lv, cnt in zip(levels, counts):
    print(f"  {lv:<25} {cnt} windows  ({cnt / len(reports) * 100:.0f}%)")